# Самостоятельная работа: задачи 1–6
По рабочей тетради 5-6. Цапков М.А. ЭПМО-01-26

Язык — Python 3. Используется только стандартная библиотека.


## 1. Реестр активных сессий

**Условие.** ID: [1201, 2304, 3415, 1201, 4526, 5637, 2304]. Построить таблицу с цепочками при m = 7.

### 1. Алгоритм и структуры данных

Таблица — список из семи цепочек. Хеш h(ID)=ID mod 7. Ищем ID в соответствующей цепочке; если отсутствует, добавляем, иначе фиксируем повтор. Храним только уникальные активные сессии.

### 2. Обоснование выбора

Цепочки позволяют хранить разные ID с одним хешем и явно показывают механизм разрешения коллизий.

### 3. Вычислительная сложность и масштабирование

Поиск и вставка в среднем O(1+α), в худшем O(N). Память O(m+U), где U — число уникальных ID. Здесь U=5, α=5/7≈0.714. Для роста данных нужно увеличивать m; при фиксированном m цепочки удлиняются.

### 4. Программная реализация


In [1]:
def session_contains(table, identifier):
    return identifier in table[identifier % len(table)]


def session_insert(table, identifier):
    bucket = table[identifier % len(table)]
    if identifier in bucket:
        return False
    bucket.append(identifier)
    return True


def build_sessions(ids, size=7):
    if size <= 0:
        raise ValueError("Размер должен быть положительным")
    table = [[] for _ in range(size)]
    duplicates = []
    for identifier in ids:
        if not session_insert(table, identifier):
            duplicates.append(identifier)
    return table, duplicates


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [2]:
ids = [1201, 2304, 3415, 1201, 4526, 5637, 2304]
table, duplicates = build_sessions(ids)
assert duplicates == [1201, 2304]
assert sum(map(len, table)) == 5
assert session_contains(table, 3415)
assert not session_contains(table, 999)
assert build_sessions([])[0] == [[] for _ in range(7)]
collision_table, _ = build_sessions([0, 7, 14])
assert collision_table[0] == [0, 7, 14]
for i, bucket in enumerate(table):
    print(i, bucket)
print("Повторы:", duplicates)
print("Коэффициент заполнения:", round(5 / 7, 3))


0 []
1 [2304]
2 [5637]
3 []
4 [1201, 4526]
5 []
6 [3415]
Повторы: [1201, 2304]
Коэффициент заполнения: 0.714


### 6.* Вывод и возможная оптимизация

При росте α таблицу следует перехешировать. Для прикладного реестра можно использовать set или dict, но здесь цепочки реализованы явно.


## 2. Дедупликация телеметрии

**Условие.** Поток: [(A,200),(B,500),(A,200),(A,404),(B,500),(C,200),(A,200)]. Сохранить первые уникальные события и посчитать коды.

### 1. Алгоритм и структуры данных

Множество хранит полные пары, список — порядок первого появления, словарь — частоты. Для ясности выводим частоты как исходного потока, так и уникального: формулировка не уточняет, на каком этапе их считать.

### 2. Обоснование выбора

Пара (устройство, код) отличает полный дубликат от другого события того же устройства. Список сохраняет порядок, который не обеспечивает множество.

### 3. Вычислительная сложность и масштабирование

В среднем O(N) времени и O(U+C) памяти для U уникальных пар и C кодов. При неограниченном потоке множество растет вместе с числом уникальных событий.

### 4. Программная реализация


In [3]:
def deduplicate_events(events):
    seen = set()
    unique = []
    raw_counts = {}
    unique_counts = {}
    for event in events:
        device, event_code = event
        raw_counts[event_code] = raw_counts.get(event_code, 0) + 1
        if event not in seen:
            seen.add(event)
            unique.append(event)
            unique_counts[event_code] = unique_counts.get(event_code, 0) + 1
    return unique, raw_counts, unique_counts


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [4]:
events = [("A", 200), ("B", 500), ("A", 200), ("A", 404), ("B", 500), ("C", 200), ("A", 200)]
unique, raw_counts, unique_counts = deduplicate_events(events)
assert unique == [("A", 200), ("B", 500), ("A", 404), ("C", 200)]
assert raw_counts == {200: 4, 500: 2, 404: 1}
assert unique_counts == {200: 2, 500: 1, 404: 1}
assert deduplicate_events([]) == ([], {}, {})
print("Уникальные:", unique)
print("Частоты исходного потока:", raw_counts)
print("Частоты после дедупликации:", unique_counts)


Уникальные: [('A', 200), ('B', 500), ('A', 404), ('C', 200)]
Частоты исходного потока: {200: 4, 500: 2, 404: 1}
Частоты после дедупликации: {200: 2, 500: 1, 404: 1}


### 6.* Вывод и возможная оптимизация

Для длительной работы следует определить окно дедупликации и срок хранения ключей, чтобы ограничить расход памяти.


## 3. Поиск сигнатуры в журнале

**Условие.** Найти все вхождения ABRA в ABCDABRAABRAEVENT.

### 1. Алгоритм и структуры данных

Рабин–Карп: считаем полиномиальный хеш сигнатуры и первого окна, затем обновляем хеш при сдвиге за O(1). При совпадении хешей обязательно сравниваем строки.

### 2. Обоснование выбора

Скользящий хеш исключает полное сравнение каждого окна. Контрольное сравнение защищает от коллизий.

### 3. Вычислительная сложность и масштабирование

Для строки длины N и шаблона длины M ожидаемое время O(N+M+KM) при K истинных совпадений и редких коллизиях; худшее O(NM). Основная дополнительная память O(1), результат O(K); срез для контрольного сравнения временно занимает O(M).

### 4. Программная реализация


In [5]:
def rabin_karp(text, pattern, modulus=1_000_000_007):
    n, m = len(text), len(pattern)
    if m == 0:
        return list(range(n + 1))
    if m > n:
        return []
    base = 257
    power = pow(base, m - 1, modulus)
    pattern_hash = window_hash = 0
    for i in range(m):
        pattern_hash = (pattern_hash * base + ord(pattern[i])) % modulus
        window_hash = (window_hash * base + ord(text[i])) % modulus
    positions = []
    for i in range(n - m + 1):
        if window_hash == pattern_hash and text[i:i + m] == pattern:
            positions.append(i)
        if i < n - m:
            window_hash = ((window_hash - ord(text[i]) * power) * base + ord(text[i + m])) % modulus
    return positions


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [6]:
positions = rabin_karp("ABCDABRAABRAEVENT", "ABRA")
assert positions == [4, 8]
assert rabin_karp("aaaa", "aa") == [0, 1, 2]
assert rabin_karp("abc", "xyz") == []
assert rabin_karp("a", "ab") == []
assert rabin_karp("", "") == [0]
# Малый модуль намеренно вызывает много коллизий.
assert rabin_karp("ABCDABRAABRAEVENT", "ABRA", modulus=1) == [4, 8]
print("Индексы с нуля:", positions)
print("Позиции с единицы:", [i + 1 for i in positions])


Индексы с нуля: [4, 8]
Позиции с единицы: [5, 9]


### 6.* Вывод и возможная оптимизация

Для гарантированного O(N+M) можно применить КМП. Двойной хеш уменьшает вероятность лишних сравнений, но не заменяет проверку совпадения.


## 4. Кэш LRU

**Условие.** Емкость 3, запросы A, B, C, A, D, B, A.

### 1. Алгоритм и структуры данных

OrderedDict хранит ключи от давно использованного к последнему. При попадании переносим ключ в конец; при промахе добавляем и при переполнении удаляем первый.

### 2. Обоснование выбора

OrderedDict объединяет быстрый поиск по ключу и изменение порядка без линейного поиска.

### 3. Вычислительная сложность и масштабирование

В среднем O(1) на запрос и O(C) памяти для емкости C. Снимки состояния в демонстрации стоят O(C) каждый и не нужны в рабочем кэше.

### 4. Программная реализация


In [7]:
from collections import OrderedDict


def simulate_lru(requests, capacity):
    if capacity < 1:
        raise ValueError("Емкость должна быть положительной")
    cache = OrderedDict()
    hits = misses = 0
    history = []
    for key in requests:
        if key in cache:
            hits += 1
            cache.move_to_end(key)
        else:
            misses += 1
            cache[key] = key
            if len(cache) > capacity:
                cache.popitem(last=False)
        history.append(list(cache))
    return hits, misses, history


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [8]:
requests = list("ABCADBA")
hits, misses, history = simulate_lru(requests, 3)
assert (hits, misses) == (2, 5)
assert history == [["A"], ["A", "B"], ["A", "B", "C"], ["B", "C", "A"], ["C", "A", "D"], ["A", "D", "B"], ["D", "B", "A"]]
assert simulate_lru([], 3) == (0, 0, [])
assert simulate_lru("ABA", 1)[:2] == (0, 3)
for key, state in zip(requests, history):
    print(key, "→", state)
print("Попаданий:", hits, "промахов:", misses)


A → ['A']
B → ['A', 'B']
C → ['A', 'B', 'C']
A → ['B', 'C', 'A']
D → ['C', 'A', 'D']
B → ['A', 'D', 'B']
A → ['D', 'B', 'A']
Попаданий: 2 промахов: 5


### 6.* Вывод и возможная оптимизация

LRU полезен при повторном обращении к недавним данным. Под нагрузкой дополнительно ограничивают размер объектов и срок жизни записей.


## 5. Диспетчер запросов

**Условие.** 501, 502, 503, 504; запрос 503 после первой обработки поставить в начало один раз.

### 1. Алгоритм и структуры данных

Дек управляет порядком, словарь хранит состояние и счетчик попыток по ID. Для 503 первая попытка заканчивается состоянием retry, вторая — done.

### 2. Обоснование выбора

Дек поддерживает оба конца за O(1), словарь позволяет проверять состояние без обхода очереди.

### 3. Вычислительная сложность и масштабирование

В среднем O(N+R) времени и O(N) памяти при R повторных постановках. Здесь 5 обработок, по две только у ID 503.

### 4. Программная реализация


In [9]:
from collections import deque


def dispatch_requests(ids, retry_id):
    queue = deque(ids)
    states = dict.fromkeys(ids, "queued")
    attempts = dict.fromkeys(ids, 0)
    order = []
    transitions = []
    while queue:
        identifier = queue.popleft()
        states[identifier] = "processing"
        attempts[identifier] += 1
        order.append(identifier)
        transitions.append((identifier, "processing"))
        if identifier == retry_id and attempts[identifier] == 1:
            states[identifier] = "retry"
            queue.appendleft(identifier)
        else:
            states[identifier] = "done"
        transitions.append((identifier, states[identifier]))
    return order, states, attempts, transitions


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [10]:
order, states, attempts, transitions = dispatch_requests([501, 502, 503, 504], 503)
assert order == [501, 502, 503, 503, 504]
assert states == {501: "done", 502: "done", 503: "done", 504: "done"}
assert attempts == {501: 1, 502: 1, 503: 2, 504: 1}
assert dispatch_requests([], 503)[0] == []
print("Порядок:", order)
print("Состояния:", states)
print("Попытки:", attempts)
print("Переходы:", transitions)


Порядок: [501, 502, 503, 503, 504]
Состояния: {501: 'done', 502: 'done', 503: 'done', 504: 'done'}
Попытки: {501: 1, 502: 1, 503: 2, 504: 1}
Переходы: [(501, 'processing'), (501, 'done'), (502, 'processing'), (502, 'done'), (503, 'processing'), (503, 'retry'), (503, 'processing'), (503, 'done'), (504, 'processing'), (504, 'done')]


### 6.* Вывод и возможная оптимизация

Счетчик ограничивает повтор и исключает бесконечный цикл. Для интеграции можно сохранять состояния и подтверждать успешную обработку в постоянном хранилище.


## 6. Комплексный мониторинг

**Условие.** Дедуплицировать записи, посчитать типы, найти два разных времени с суммой 630 мс и сформировать очередь.

### 1. Алгоритм и структуры данных

Множество полных записей устраняет повторы, список сохраняет порядок, словарь считает типы. Для каждой уникальной записи ищем дополнение 630−time во множестве уже просмотренных времен. Готовый уникальный поток помещаем в deque. Частоты выводим до и после дедупликации.

### 2. Обоснование выбора

Полная запись является ключом дедупликации. Поиск дополнения заменяет перебор пар; условие time != complement обеспечивает два различных значения времени.

### 3. Вычислительная сложность и масштабирование

В среднем O(N) времени, O(U) памяти. При уникальных данных U=N; точная дедупликация всего бесконечного потока требует неограниченной памяти.

### 4. Программная реализация


In [11]:
def monitor(records, target=630):
    seen = set()
    times = set()
    queue = deque()
    raw_counts = {}
    unique_counts = {}
    pair = None
    for record in records:
        user, kind, time = record
        raw_counts[kind] = raw_counts.get(kind, 0) + 1
        if record in seen:
            continue
        seen.add(record)
        queue.append(record)
        unique_counts[kind] = unique_counts.get(kind, 0) + 1
        complement = target - time
        if pair is None and complement != time and complement in times:
            pair = (complement, time)
        times.add(time)
    return queue, raw_counts, unique_counts, pair


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [12]:
records = [("u1", "login", 120), ("u2", "read", 80), ("u1", "login", 120), ("u3", "error", 450), ("u2", "read", 80), ("u4", "error", 510)]
queue, raw_counts, counts, pair = monitor(records)
assert list(queue) == [records[0], records[1], records[3], records[5]]
assert raw_counts == {"login": 2, "read": 2, "error": 2}
assert counts == {"login": 1, "read": 1, "error": 2}
assert pair == (120, 510)
assert monitor([])[3] is None
assert monitor([("a", "x", 315), ("b", "x", 315)])[3] is None
print("Очередь уникальных записей:", list(queue))
print("Частоты исходного потока:", raw_counts)
print("Частоты уникальных записей:", counts)
print("Пара времен:", pair, "сумма:", sum(pair))


Очередь уникальных записей: [('u1', 'login', 120), ('u2', 'read', 80), ('u3', 'error', 450), ('u4', 'error', 510)]
Частоты исходного потока: {'login': 2, 'read': 2, 'error': 2}
Частоты уникальных записей: {'login': 1, 'read': 1, 'error': 2}
Пара времен: (120, 510) сумма: 630


### 6.* Вывод и возможная оптимизация

Решение масштабируется линейно по потоку. При длительной работе нужны временные окна и ограничение очереди; при разделении по узлам одинаковые ключи должны попадать в одну область дедупликации.
